In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from src.trainer.BufferTrainer import BufferTrainer
from src.trainer.SimpleTrainer import SimpleTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils import InContextHead
from src import models
from src.buffer import MultiTaskBuffer

### Task Incremental Training

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cpu")
head.set_context(0)
model = models.get_fully_connected_mnist_model(head=head)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

In [ ]:
buffer = MultiTaskBuffer([])
interval_trainer = BufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.9,
    primal_learning_rate=0.5,
    paradigm="TIL",
    buffer=buffer,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    interval_trainer.train(train, val, batch_size=256, context_id=i + 1, target_acc=0.9)
    interval_trainer.test(test_tasks[0 : i + 2], context_list=list(range(i + 2)))
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test, context_id=i + 1)

### Domain Incremental Training

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=0)

model = models.get_mnist_model(seed=0)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
train_tasks, buffer_sets = zip(
    *[create_holdout_set(dataset) for dataset in train_tasks]
)
print([len(task) for task in buffer_sets])

In [ ]:
trainer = SimpleTrainer(model)
trainer.train(
    train_tasks[0], val_tasks[0], epochs=3, batch_size=64, domain_map_fn=domain_map_fn
)
trainer.test(test_tasks[0:1], domain_map_fn=domain_map_fn)

In [ ]:
buffer = MultiTaskBuffer([])
interval_trainer = BufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=20,
    min_acc_limit=0.9,
    primal_learning_rate=0.5,
    paradigm="DIL",
    buffer=buffer,
    domain_map_fn=domain_map_fn,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])
interval_trainer.add_to_buffer(buffer_sets[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, context_id=i, target_acc=0.9)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)
        interval_trainer.add_to_buffer(buffer_sets[i])

### Class Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [ ]:
train_tasks, buffer_sets = zip(
    *[create_holdout_set(dataset) for dataset in train_tasks]
)
print([len(task) for task in buffer_sets])

[3837, 3839, 3838, 3839, 3844]


In [9]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████| 3/3 [00:40<00:00, 13.43s/it, val_loss=0.0131, val_acc=0.996]


Test Results: [(0.0142, 0.9955)]


In [11]:
buffer = MultiTaskBuffer([])
interval_trainer = BufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=20,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="CIL",
    buffer=buffer,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])
interval_trainer.add_to_buffer(buffer_sets[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, target_acc=0.9)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)
        interval_trainer.add_to_buffer(buffer_sets[i])

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1961 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=1.00


100%|██████████| 20/20 [00:15<00:00,  1.31it/s, size=42.29, obj=0.006, min_soft_acc=0.909]


Final bbox:  Obj=0.01,  Size=42.29,  Min acc hard=0.95,  Min acc soft=0.94
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 1 checkpoints
Checkpoints sizes: ['42.29']
Checkpoint certificates: ['0.95']
----------------------- Finished Computing Rashomon set ------------------------


Processing items: 1it [00:02,  2.23s/it, Training loss=32.5, Training accuracy=0, no_improvement=9]

Using buffer to continue training.
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1957 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=1.00


...

Exiting with final training accuracy of 0.0000
Test Results: [(0.5979, 0.8025), (0.4749, 0.8644), (28.3621, 0.0)]
---------------------------- Computing Rashomon set ----------------------------


Processing items: 1it [00:02,  2.55s/it, Training loss=31.9, Training accuracy=0, no_improvement=9]


Exiting with final training accuracy of 0.0000
Test Results: [(0.5979, 0.8025), (0.4749, 0.8644), (29.5975, 0.0), (32.0183, 0.0)]


Processing items: 1it [00:02,  2.35s/it, Training loss=24.6, Training accuracy=0, no_improvement=9]


Exiting with final training accuracy of 0.0000
Test Results: [(0.5979, 0.8025), (0.4749, 0.8644), (31.0957, 0.0), (35.2029, 0.0), (25.5282, 0.0)]
